# Delete Exposures and Nights

Interactively delete exposures and nights of observations after inspect_nights.

Author: Qiushi Chris Tian

Created: 2026-09-28

<!-- Updated: -->

## Setting/Config

In [39]:
TARGET = 'TOI-431'

WHOLE_NIGHTS = [20231015, ]
BAND_NIGHTS = {20240208: 'I', }
JDS = [2460237.8099118057, ]

print(
    f'{len(WHOLE_NIGHTS)} whole night(s), '
    f'{len(BAND_NIGHTS)} band night(s), and '
    f'{len(JDS)} individual exposure(s)'
)

PREVIEW_COLS = ['band', 'night', 'time', 'jd', 'airmass', 'exptime', 'skytemp']

1 whole night(s), 1 band night(s), and 1 individual exposure(s)


## Preamble

In [2]:
# %matplotlib ipympl

In [64]:
# from dep import plt, va
import numpy as np
from pathlib import Path
from astropy.table import Table
from newport import get_input_path

In [9]:
input_path = get_input_path(TARGET)
print(f'Reading original table from: {input_path}\n')

t = Table.read(input_path)
print(f'Original length: {len(t)}\n')

Reading original table from: data/tables/list_runs/TOI-431/phot_list_run/phot_w_err_TOI-431.fits

Original length: 3306

Modified table will be saved to: data/tables/opt_comp_stars/TOI-431/phot_w_err_TOI-431_manual_del.fits



## Building Mask

In [62]:
del_mask = np.zeros(len(ori_table), dtype=bool)

for wn in WHOLE_NIGHTS:
    selected = t['night'] == wn
    print(f'{np.sum(selected)} exposure(s) on {wn} ...')
    del_mask |= selected

for night, bands in BAND_NIGHTS.items():
    for band in bands:
        selected = (t['band'] == band) & (t['night'] == night)
        print(f'{np.sum(selected)} exposure(s) in {band} band on {night} ...')
        del_mask |= selected

for jd in JDS:
    selected = t['jd'] == jd
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

print('Done!')

14 exposure(s) on 20231015 ...
20 exposure(s) in I band on 20240208 ...
1 exposure on 20231019 ...
Done!


## Preview Deletion

In [52]:
dt = t[del_mask]

print(f'Number of exposure(s): {len(dt)}')
print(f'Unique night(s): {len(np.unique(dt['night']))}\n')

dt[PREVIEW_COLS].pprint_all()

print(f'\nAfter deletion, length will become {len(t) - len(dt)}')

Number of exposure(s): 35
Unique night(s): 3

band  night             time                  jd            airmass    exptime skytemp
---- -------- ----------------------- ------------------ ------------- ------- -------
   B 20231015 2023-10-16T09:36:09.910 2460233.9001146993 2.80353101415    10.0   -13.5
   B 20231015 2023-10-16T09:36:55.490 2460233.9006422455 2.80742818989    10.0   -13.5
   V 20231015 2023-10-16T09:33:23.789  2460233.898192014 2.79026081128    10.0   -12.3
   V 20231015 2023-10-16T09:32:36.539 2460233.8976451387 2.78663602129    10.0   -13.3
   B 20231015 2023-10-16T09:36:32.600  2460233.900377315 2.80544345889    10.0   -13.5
   B 20231015 2023-10-16T09:37:19.320 2460233.9009180553 2.80942701658    10.0   -12.7
   V 20231015 2023-10-16T09:33:00.350 2460233.8979207175 2.78843912189    10.0   -13.1
   V 20231015 2023-10-16T09:32:13.519 2460233.8973787036 2.78483899711    10.0   -13.2
   V 20231015 2023-10-16T09:35:19.910 2460233.8995359954 2.79947777151    10.0   -12

## Executing Deletion

In [50]:
new_t = t[~del_mask]

print(f'New length: {len(new_t)}')

New length: 3271


## Save Table

In [ ]:
save_path = f'data/tables/opt_comp_stars/{TARGET}/{input_path.stem}_manual_del{input_path.suffix}'
print(f'Modified table will be saved to: {save_path}\n')

In [65]:
if not Path(save_path).exists() or input(f'{save_path} exists. Overwrite? (y/*)').casefold().startswith('y'):
    new_t.write(save_path, overwrite=True)
    print(f'Saved to {save_path}')
else:
    print(f'Saving aborted')

KeyboardInterrupt: Interrupted by user